In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/PrepaidTravelCost.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/2099 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18246 entries, 0 to 18245
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id                                18246 non-null  object             
 1   org:resource                      18246 non-null  object             
 2   concept:name                      18246 non-null  object             
 3   time:timestamp                    18246 non-null  datetime64[ns, UTC]
 4   org:role                          18246 non-null  object             
 5   case:Rfp_id                       18246 non-null  object             
 6   case:Permit travel permit number  18246 non-null  object             
 7   case:Task                         18246 non-null  object             
 8   case:OrganizationalEntity         18246 non-null  object             
 9   case:RequestedAmount              18246 non-null  float64    

In [6]:
df.isnull().any()

id                                  False
org:resource                        False
concept:name                        False
time:timestamp                      False
org:role                            False
case:Rfp_id                         False
case:Permit travel permit number    False
case:Task                           False
case:OrganizationalEntity           False
case:RequestedAmount                False
case:Activity                       False
case:Permit TaskNumber              False
case:Permit BudgetNumber            False
case:Permit ProjectNumber           False
case:Project                        False
case:concept:name                   False
case:Permit OrganizationalEntity    False
case:Permit RequestedBudget         False
case:Cost Type                      False
case:Permit id                      False
case:Permit ActivityNumber          False
case:RfpNumber                      False
dtype: bool

In [7]:
df = df.drop(columns=['case:Rfp_id', 'case:Permit travel permit number', 'case:Permit TaskNumber', 'case:Permit BudgetNumber', 
                      'case:Permit ProjectNumber', 'case:Permit id', 'case:Permit ActivityNumber', 'case:RfpNumber', 'case:Cost Type', 'id',
                      'case:Task', 'case:Project'])

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['org:resource'] = df['org:resource'].astype('string')
df['org:role'] = df['org:role'].astype('string')
df['case:Activity'] = df['case:Activity'].astype('string')
df['case:OrganizationalEntity'] = df['case:OrganizationalEntity'].astype('string')

df['case:RequestedAmount'] = df['case:RequestedAmount'].astype(np.float32)
df['case:Permit RequestedBudget'] = df['case:Permit RequestedBudget'].astype(np.float32)

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds().astype(np.float32)
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [12]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
6358,request for payment 1000,2018-03-01 10:55:17+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
6359,request for payment 1000,2018-03-01 10:55:21+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
6360,request for payment 1000,2018-03-01 11:34:16+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,2335.0
6361,request for payment 1000,2018-03-01 11:34:23+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,7.0
6362,request for payment 1000,2018-03-01 15:01:48+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,12445.0
6363,request for payment 1000,2018-03-05 14:49:53+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,344885.0
6364,request for payment 1000,2018-03-06 10:13:29+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request Payment,SYSTEM,UNDEFINED,69816.0
6365,request for payment 1000,2018-03-08 17:31:00+00:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Payment Handled,SYSTEM,UNDEFINED,199051.0
5823,request for payment 10043,2018-02-20 13:53:11+00:00,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
5824,request for payment 10043,2018-02-20 13:53:14+00:00,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0


In [13]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 2099


### --- Feature Configurations ---

In [14]:
# --- Define feature specs ---
feature_specs = {

    "time_delta": {
        "type":           "continuous",
        "level":          "event",
        "vary":           True,
        "quantile_low":   0.20,
        "quantile_high":  0.80, 
    },

    "case:RequestedAmount": {
        "type":           "continuous",
        "level":          "case",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
    },

    "case:Permit RequestedBudget": {
        "type":           "continuous",
        "level":          "case",
        "vary":           True,
        "quantile_low":   0.05,
        "quantile_high":  0.90, 
    },

    "org:resource": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "org:role": {
        "type":           "categorical",
        "level":          "event",
        "vary":           True,
    },

    "case:Activity": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    "case:OrganizationalEntity": {
        "type":           "categorical",
        "level":          "case",
        "vary":           True,
    },

    # immutable
    "concept:name": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [15]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [16]:
# feature_config = FeatureConfig.load()

In [17]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [6.00, 362277.00]                        61266.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [64.68, 1661.05]                         338.4588   quantile_derived    
case:Permit RequestedBudget    continuous     case     yes    [130.85, 4066.04]                        769

### --- Next event prediction model ---

In [18]:
# Transform nan cols to NA
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [19]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [20]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [21]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [22]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['org:resource', 'org:role', 'case:Activity', 'case:OrganizationalEntity', 'concept:name']
  activity_prototypes: 29 activities


In [23]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [24]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=train_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    df=val_df,
    case_id_field="case:concept:name", 
    sort_field="time:timestamp"
)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [26]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 29, 'org:resource': 2, 'org:role': 8}, 'static_categorical_info': {'case:Activity': 5, 'case:OrganizationalEntity': 26}}


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
device

device(type='cuda')

In [29]:
criterion = torch.nn.CrossEntropyLoss()

In [30]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Ptc-model_output.txt")

Epoch 020/100 | Train Loss: 0.2758 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.2279 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.1980 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.1789 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.1733 | LR: 1.00e-06
Time taken for next event model (training): 214.362657 seconds
Time taken for next event model (validation): 0.187118 seconds
Val loss: {'loss': 0.4520305660492044, 'accuracy': 0.8750782717595491, 'f1_macro': 0.6246817179046388, 'f1_weighted': 0.8723015663207095}


In [31]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [32]:
# model = ProcessLSTM.load()

In [33]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [34]:
# --- Save processed df ---
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/bpic20_Ptc.xlsx", index=False, engine="openpyxl")

In [35]:
sys.stdout = original_stdout
log_file.close()